# UniAdvisor: chat messages for training the message classifier

Writes Vietnamese messages a grade-12 student (or a parent) might type into UniAdvisor's chat, labelled with one of 19
request types (`src/uniadvisor/chat/types.py`). The labels are known by construction: each prompt asks for messages of
one given type.

**Method (cited, verified 2026-10-06)**
- Prompting off-the-shelf LLMs with a description and seed examples of one intent to generate labelled training data:
  Sahu et al. 2022, *Data Augmentation for Intent Classification with Off-the-shelf Large Language Models*,
  4th Workshop on NLP for Conversational AI (ACL 2022), pp. 47-57.
- Attributed prompts (each prompt samples style, length, accents, typos, persona) for diversity and less bias:
  Yu et al. 2023, *Large Language Model as Attributed Training Data Generator: A Tale of Diversity and Bias*,
  NeurIPS 2023 Datasets and Benchmarks.
- Survey of generation, curation and evaluation: Long et al. 2024, *On LLMs-Driven Synthetic Data Generation,
  Curation, and Evaluation: A Survey*, Findings of ACL 2024, pp. 11065-11082.
- Several LLM families, and a held-out family for testing. Families here: Qwen 2.5 (Alibaba), Aya Expanse
  (Cohere), Gemma 2 (Google) for training; SeaLLM-7B-v2 (DAMO Academy, built on Mistral 7B, supports Vietnamese)
  for testing. Vistral-7B-Chat, the first choice, needs a manual approval by its authors that did not come.
  SeaLLM's outputs are used only as test messages, never to train or improve a language model (SeaLLMs License
  Agreement); notice: "SeaLLMs is licensed under the SeaLLMs LICENSE AGREEMENT, Copyright (c) Damo Academy."

**Before running (Kaggle)**
1. Settings: Accelerator *GPU T4 x2*, Internet *on*.
2. Gated models: with your Hugging Face account, accept the licences of `CohereLabs/aya-expanse-8b`,
   `google/gemma-2-9b-it` (SeaLLM and Qwen are not gated), then add a Kaggle secret named `HF_TOKEN` (Add-ons ->
   Secrets). Qwen is not gated. Aya Expanse is CC-BY-NC (non-commercial use only).
3. Choose the families to run in `RUN_FAMILIES` (one or two per session keeps you inside Kaggle's time limit).

**Output**: `/kaggle/working/chat_generated_<family>.csv` (round 2: `<family>_round2`) (columns `text, type, family, model, attributes, prompt_id`)
and `chat_review_sample_<family>.csv` (20 messages per type for people to check). Download them into the repo's
`pipelines/data/chat/generated/` (review samples into `pipelines/data/chat/review/`), then run `python -m pipelines.chat_model.train`.

In [ ]:
# ---- settings ----------------------------------------------------------------------------------------------
# Round 2 (2026-10-08): only the three types added after round 1, for every family (round 1 files are kept).
RUN_FAMILIES = ["qwen", "aya", "gemma", "mistral"]
RUN_TYPES = ["favorite_school", "own_score", "cutoff_shift"]   # None: every type
ROUND = 2                      # in the output file names, so round 1's files are not overwritten
PROMPTS_PER_TYPE = 40          # prompts per request type and family
MESSAGES_PER_PROMPT = 5        # messages asked for in one prompt
BATCH_SIZE = 8
MAX_NEW_TOKENS = 400
TEMPERATURE, TOP_P = 0.9, 0.95
SEED = 2027

MODELS = {
    # family: (model id, how to load). fp16 on T4 (no bf16); Gemma 2 overflows in fp16, so it runs 4-bit with fp32 compute.
    "qwen":    ("Qwen/Qwen2.5-7B-Instruct", "fp16"),
    "aya":     ("CohereLabs/aya-expanse-8b", "fp16"),
    "gemma":   ("google/gemma-2-9b-it", "4bit_fp32"),
    "mistral": ("SeaLLMs/SeaLLM-7B-v2", "fp16"),        # held-out test family (Mistral 7B base)
}

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import gc, json, os, random, re, time
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:   # not on Kaggle, or no secret: only ungated models will load
    print("No HF_TOKEN secret:", e)
print(torch.cuda.device_count(), "GPU(s)")

## Names, request types, seed examples and attributes (from UniAdvisor's data)

In [ ]:
SLOTS = {
 "schools": [
  "AOF",
  "An ninh Nhân dân",
  "Biên phòng",
  "Bà Rịa - Vũng Tàu",
  "Bách Khoa - Đại học Quốc gia TP.HCM",
  "Bách Khoa Hà Nội",
  "Bách khoa TP.HCM",
  "Báo chí và Tuyên truyền",
  "Bình Dương",
  "Bưu chính",
  "Chính sách và Phát triển",
  "Chính trị Công an Nhân dân",
  "Cán bộ thành phố HCM",
  "Công Thương Tp.HCM",
  "Công nghiệp Dệt may Hà Nội",
  "Công nghiệp Hà Nội",
  "Công nghiệp Việt - Hung",
  "Công nghiệp tp. Hồ Chí Minh",
  "Công nghệ - Đại học Quốc gia Hà Nội",
  "Công nghệ Bưu chính Viễn thông",
  "Công nghệ Bưu chính Viễn thông - Cơ sở 2",
  "Công nghệ Giao thông Vận tải",
  "Công nghệ Sài Gòn",
  "Công nghệ Thông tin - Đại học Quốc gia TP.HCM",
  "Công nghệ Tp. Hồ Chí Minh",
  "Công nghệ và Quản lý Hữu nghị",
  "Công Đoàn",
  "Cảnh sát Nhân dân",
  "DAV",
  "Dân lập Phương Đông",
  "Dân lập Đông Đô",
  "Dược Hà Nội",
  "Dầu khí Việt Nam",
  "FTU",
  "Fpt",
  "Gia Định",
  "Giao thông Vận tải",
  "Giao thông Vận tải - Cơ sở 2 ở phía Nam",
  "Giao thông Vận tải tp. Hồ Chí Minh",
  "Giáo dục - Đại học Quốc gia Hà Nội",
  "HANU",
  "HCMUS",
  "HCMUT",
  "HCMUTE",
  "HLU",
  "HMU",
  "HNUE",
  "HOU",
  "HUS",
  "HUST",
  "HaUI",
  "Hoa Sen",
  "Hà Nội",
  "Hàng không Việt Nam",
  "Hành chính Quốc gia",
  "Hành chính Quốc gia - Cơ sở phía Nam",
  "Hòa Bình",
  "Hậu cần (hệ Dân sự)",
  "Hậu cần (hệ Quân sự)",
  "IU",
  "Khoa Quốc tế - Đại học Quốc gia Hà Nội",
  "Khoa Y - Dược - Đại học Quốc gia Hà Nội",
  "Khoa học Quân sự (dân sự)",
  "Khoa học Quân sự (quân sự)",
  "Khoa học Sức khỏe - ĐHQG Tp.HCM",
  "Khoa học Tự nhiên - Đại học Quốc gia Hà Nội",
  "Khoa học Tự nhiên - Đại học Quốc gia TP.HCM",
  "Khoa học Xã hội và Nhân văn - Đại học Quốc gia Hà Nội",
  "Khoa học Xã hội và Nhân văn - Đại học Quốc gia TP.HCM",
  "Khoa luật - Đại học Quốc gia Hà Nội",
  "Kinh doanh và Công nghệ Hà Nội",
  "Kinh tế - Kỹ thuật Bình Dương",
  "Kinh tế - Luật - Đại học Quốc gia TP.HCM",
  "Kinh tế - Tài chính tp. Hồ Chí Minh",
  "Kinh tế - Đại học Quốc gia Hà Nội",
  "Kinh tế Kỹ thuật Công nghiệp",
  "Kinh tế Quốc dân",
  "Kinh tế tp. Hồ Chí Minh",
  "Kiến trúc Thành phố Hồ Chí Minh",
  "Kiểm sát Hà Nội",
  "Kỹ thuật Mật mã",
  "Kỹ thuật Quân sự (hệ Dân sự)",
  "Kỹ thuật Quân sự (hệ Quân sự)",
  "Lao động - Xã hội (Cơ sở II TP. HCM)",
  "Lao động - Xã hội (Cơ sở Sơn Tây)",
  "Lao động - Xã hội (Trụ sở chính)",
  "Luật Hà Nội",
  "Luật tp. Hồ Chí Minh",
  "Lâm nghiệp (Phía Bắc)",
  "Mỏ - Địa chất",
  "Mở Hà Nội",
  "Mở tp. Hồ Chí Minh",
  "Mỹ thuật Công nghiệp",
  "Mỹ thuật Công nghiệp Á Châu",
  "Mỹ thuật Việt Nam",
  "Mỹ thuật tp. Hồ Chí Minh",
  "NEU",
  "Ngoại giao",
  "Ngoại ngữ - Tin học TP.HCM",
  "Ngoại ngữ - Đại học Quốc gia Hà Nội",
  "Ngoại thương",
  "Ngoại thương - Cơ sở phía Nam",
  "Nguyễn Trãi",
  "Nguyễn Tất Thành",
  "Ngân hàng (Cơ sở Hà Nội)",
  "Ngân hàng tp. Hồ Chí Minh",
  "Nhạc viện Tp. Hồ Chí Minh",
  "Nông lâm Tp. Hồ Chí Minh",
  "Nông nghiệp Việt Nam",
  "Nội vụ Hà Nội",
  "PTIT",
  "Phòng cháy Chữa cháy",
  "Phòng không - Không quân",
  "Phụ nữ Việt Nam",
  "Quân Y (Dân sự)",
  "Quân y (Hệ quân sự)",
  "Quản Trị và Kinh Doanh - ĐH Quốc gia Hà Nội",
  "Quản lý Giáo dục",
  "Quốc tế - Đại học Quốc gia TP.HCM",
  "Quốc tế Sài Gòn",
  "Quốc tế miền Đông",
  "Sài Gòn",
  "Sân khấu Điện ảnh Hà Nội",
  "Sân khấu, Điện ảnh tp. Hồ Chí Minh",
  "Sĩ quan Công binh",
  "Sĩ quan Kỹ thuật quân sự",
  "Sĩ quan Pháo binh",
  "Sĩ quan Đặc công",
  "Sư phạm Hà Nội",
  "Sư phạm Kỹ thuật tp. Hồ Chí Minh",
  "Sư phạm Nghệ thuật Trung ương",
  "Sư phạm Thể dục Thể thao Hà Nội",
  "Sư phạm Thể dục Thể thao TP.HCM",
  "Sư phạm tp. Hồ Chí Minh",
  "TDTU",
  "TLU",
  "Thanh thiếu niên Việt Nam",
  "Thuỷ lợi",
  "Thuỷ lợi - Cơ sở 2 ở phía Nam",
  "Thành Đô",
  "Thăng Long",
  "Thương mại",
  "Thể dục Thể thao Thành phố Hồ Chí Minh",
  "Thủ Dầu Một",
  "Thủ đô Hà Nội",
  "Toà án",
  "Trưng vương",
  "Trần Quốc Tuấn",
  "Trần Đại Nghĩa (hệ dân sự)",
  "Tài chính",
  "Tài chính - Marketing",
  "Tài chính - Ngân hàng Hà Nội",
  "Tài nguyên và Môi Trường Hà Nội",
  "Tài nguyên và Môi Trường tp. Hồ Chí Minh",
  "Tôn Đức Thắng",
  "UEB",
  "UEH",
  "UEL",
  "UET",
  "UIT",
  "UMP",
  "USSH",
  "UTC",
  "VNUA",
  "Việt Đức",
  "Văn Hiến",
  "Văn Lang",
  "Văn hoá - Nghệ thuật Quân đội",
  "Văn hoá Hà Nội",
  "Văn hoá Tp. Hồ Chí Minh",
  "Xây dựng",
  "Y Hà Nội",
  "Y dược TP.HCM",
  "Y dược học Cổ truyền Việt Nam",
  "Y khoa Phạm Ngọc Thạch",
  "Y tế Công cộng",
  "sĩ quan Phòng hoá",
  "Âm nhạc Quốc gia Việt Nam",
  "Điện lực",
  "Đại Học CMC",
  "Đại Học Khoa Học Và Công Nghệ Hà Nội",
  "Đại Học Phenikaa",
  "Đại Học Phòng Cháy Chữa Cháy ( Hệ Dân sự Phía Bắc )",
  "Đại Nam",
  "Đại học Kiến trúc Hà Nội",
  "Đại học Phòng cháy Chữa cháy phía Nam",
  "Đại học Quốc tế Hồng Bàng"
 ],
 "nganh": [
  "An ninh mạng",
  "An ninh mạng và phòng chống tội phạm công nghệ cao",
  "An toàn",
  "An toàn thông tin",
  "An toàn, Vệ sinh lao động",
  "Biên phòng",
  "Biên đạo múa",
  "Biến đổi khí hậu",
  "Biến đổi khí hậu và phát triển bền vững",
  "Biểu diễn nhạc cụ phương tây",
  "Biểu diễn nhạc cụ truyền thống",
  "Báo chí",
  "Bảo dưỡng công nghiệp",
  "Bảo hiểm",
  "Bảo hiểm - Tài chính",
  "Bảo hộ lao động",
  "Bảo tàng học",
  "Bảo vệ thực vật",
  "Bất động sản",
  "Chính trị học",
  "Chăn nuôi",
  "Chăn nuôi thú y - Thuỷ sản",
  "Chỉ huy kỹ thuật Phòng không",
  "Chỉ huy kỹ thuật công binh",
  "Chỉ huy kỹ thuật hóa học",
  "Chỉ huy tham mưu Lục quân",
  "Chỉ huy tham mưu Pháo binh",
  "Chỉ huy tham mưu Đặc công",
  "Chỉ huy âm nhạc",
  "Chỉ huy, quản lý kỹ thuật",
  "Chủ nghĩa xã hội khoa học",
  "Công nghệ Vật lý điện tử và tin học",
  "Công nghệ bán dẫn",
  "Công nghệ chế biến lâm sản",
  "Công nghệ chế biến thủy sản",
  "Công nghệ chế tạo máy",
  "Công nghệ dược phẩm",
  "Công nghệ dệt",
  "Công nghệ dệt, may",
  "Công nghệ giáo dục",
  "Công nghệ hàng không vũ trụ",
  "Công nghệ hóa học",
  "Công nghệ kỹ thuật công trình xây dựng",
  "Công nghệ kỹ thuật cơ khí",
  "Công nghệ kỹ thuật cơ điện tử",
  "Công nghệ kỹ thuật giao thông",
  "Công nghệ kỹ thuật hóa học",
  "Công nghệ kỹ thuật hạt nhân",
  "Công nghệ kỹ thuật in",
  "Công nghệ kỹ thuật máy tính",
  "Công nghệ kỹ thuật môi trường",
  "Công nghệ kỹ thuật nhiệt",
  "Công nghệ kỹ thuật năng lượng",
  "Công nghệ kỹ thuật năng lượng tái tạo",
  "Công nghệ kỹ thuật vật liệu xây dựng",
  "Công nghệ kỹ thuật xây dựng",
  "Công nghệ kỹ thuật ô tô",
  "Công nghệ kỹ thuật điều khiển và tự động hóa",
  "Công nghệ kỹ thuật điện",
  "Công nghệ kỹ thuật điện tử - viễn thông",
  "Công nghệ kỹ thuật điện, điện tử",
  "Công nghệ may",
  "Công nghệ nông nghiệp",
  "Công nghệ sinh học",
  "Công nghệ số trong thăm dò và khai thác tài nguyên thiên nhiên",
  "Công nghệ sợi, dệt",
  "Công nghệ thông tin",
  "Công nghệ thẩm mỹ",
  "Công nghệ thực phẩm",
  "Công nghệ truyền thông",
  "Công nghệ tài chính",
  "Công nghệ tài chính (Fintech)",
  "Công nghệ vật liệu",
  "Công nghệ vật liệu dệt, may",
  "Công nghệ ô tô điện",
  "Công nghệ đa phương tiện",
  "Công nghệ điện ảnh, truyền hình",
  "Công trình giao thông",
  "Công tác thanh thiếu niên",
  "Công tác xã hội",
  "Cơ kỹ thuật",
  "Cảnh quan và kỹ thuật hoa viên",
  "Di sản học",
  "Digital Art (Nghệ thuật số)",
  "Digital Marketing",
  "Dinh dưỡng",
  "Dinh dưỡng và Khoa học thực phẩm",
  "Dinh dưỡng và chế biến món ăn",
  "Du lịch",
  "Du lịch sinh thái",
  "Du lịch địa chất",
  "Dược học",
  "Giáo dục Chính trị",
  "Giáo dục Công dân",
  "Giáo dục Mầm non",
  "Giáo dục Quốc phòng - An ninh",
  "Giáo dục Tiểu học",
  "Giáo dục học",
  "Giáo dục Đặc biệt",
  "Giới và Phát triển",
  "Hoa Kỳ học",
  "Hàn",
  "Hàn Quốc học",
  "Hán Nôm",
  "Hình ảnh y học",
  "Hóa dược",
  "Hóa học",
  "Hải dương học",
  "Hậu cần quân sự",
  "Hệ thống kỹ thuật công trình xây dựng",
  "Hệ thống nhúng và IoT",
  "Hệ thống thông tin",
  "Hệ thống thông tin quản lý",
  "Hỗ trợ giáo dục người khuyết tật",
  "Hộ sinh",
  "Hội họa",
  "Khai thác và kinh tế vận tải",
  "Khai thác vận tải",
  "Khoa học chế biến món ăn",
  "Khoa học cây trồng",
  "Khoa học dinh dưỡng và ẩm thực",
  "Khoa học dữ liệu",
  "Khoa học máy tính",
  "Khoa học máy tính và thông tin",
  "Khoa học môi trường",
  "Khoa học quản lý",
  "Khoa học thông tin địa không gian",
  "Khoa học tính toán",
  "Khoa học và công nghệ thực phẩm",
  "Khoa học vật liệu",
  "Khoa học y sinh",
  "Khí tượng - thủy văn",
  "Khí tượng và khí hậu học",
  "Khúc xạ nhãn khoa",
  "Kinh doanh nông nghiệp",
  "Kinh doanh quốc tế",
  "Kinh doanh thương mại",
  "Kinh doanh thời trang và dệt may",
  "Kinh doanh xuất bản phẩm",
  "Kinh tế",
  "Kinh tế chính trị",
  "Kinh tế công nghiệp",
  "Kinh tế nông nghiệp",
  "Kinh tế phát triển",
  "Kinh tế quốc tế",
  "Kinh tế số",
  "Kinh tế thể thao",
  "Kinh tế tài nguyên thiên nhiên",
  "Kinh tế vận tải",
  "Kinh tế xây dựng",
  "Kinh tế đầu tư",
  "Kiến trúc",
  "Kiến trúc cảnh quan",
  "Kiến trúc đô thị",
  "Kiểm toán",
  "Kế toán",
  "Kỹ nghệ gỗ và nội thất",
  "Kỹ thuật Bán dẫn",
  "Kỹ thuật Gây mê hồi sức",
  "Kỹ thuật Phục hồi chức năng",
  "Kỹ thuật Robot",
  "Kỹ thuật Robot và Điều khiển thông minh",
  "Kỹ thuật an toàn giao thông",
  "Kỹ thuật công nghiệp",
  "Kỹ thuật cơ khí",
  "Kỹ thuật cơ khí động lực",
  "Kỹ thuật cơ sở hạ tầng",
  "Kỹ thuật cơ điện tử",
  "Kỹ thuật cấp thoát nước",
  "Kỹ thuật dầu khí",
  "Kỹ thuật dệt",
  "Kỹ thuật dữ liệu",
  "Kỹ thuật hoá học",
  "Kỹ thuật hàng không",
  "Kỹ thuật hình ảnh y học",
  "Kỹ thuật hạt nhân",
  "Kỹ thuật hệ thống công nghiệp",
  "Kỹ thuật in",
  "Kỹ thuật khí thiên nhiên",
  "Kỹ thuật không gian",
  "Kỹ thuật máy tính",
  "Kỹ thuật môi trường",
  "Kỹ thuật mỏ",
  "Kỹ thuật nhiệt",
  "Kỹ thuật năng lượng",
  "Kỹ thuật phần mềm",
  "Kỹ thuật phục hình răng",
  "Kỹ thuật sinh học",
  "Kỹ thuật thực phẩm",
  "Kỹ thuật trắc địa - bản đồ",
  "Kỹ thuật tuyển khoáng",
  "Kỹ thuật tài nguyên nước",
  "Kỹ thuật tàu thủy",
  "Kỹ thuật vật liệu",
  "Kỹ thuật xây dựng",
  "Kỹ thuật xây dựng công trình giao thông",
  "Kỹ thuật xây dựng công trình thủy",
  "Kỹ thuật xét nghiệm y học",
  "Kỹ thuật y sinh",
  "Kỹ thuật ô tô",
  "Kỹ thuật ô tô và năng lượng mới",
  "Kỹ thuật Điện và Máy tính (ECE)",
  "Kỹ thuật Đường sắt",
  "Kỹ thuật điều khiển và tự động hóa",
  "Kỹ thuật điện",
  "Kỹ thuật điện tử - viễn thông",
  "Kỹ thuật điện tử và tin học",
  "Kỹ thuật địa chất",
  "Kỹ thuật địa vật lý",
  "Logistics và Quản lý chuỗi cung ứng",
  "Luật",
  "Luật dân sự và tố tụng dân sự",
  "Luật hình sự và tố tụng hình sự",
  "Luật kinh tế",
  "Luật quốc tế",
  "Luật thương mại quốc tế",
  "Lâm học",
  "Lâm nghiệp",
  "Lâm nghiệp đô thị",
  "Lâm sinh",
  "Lý luận, lịch sử và phê bình mỹ thuật",
  "Lý luận, lịch sử và phê bình điện ảnh, truyền hình",
  "Lưu trữ học",
  "Lịch sử",
  "Marketing",
  "Môi trường",
  "Môi trường, sức khỏe và an toàn",
  "Mạng máy tính và truyền thông dữ liệu",
  "Nghiên cứu phát triển",
  "Nghiệp vụ an ninh; Nghiệp vụ cảnh sát",
  "Nghệ thuật học",
  "Nghệ thuật số",
  "Nghệ thuật trình diễn",
  "Ngôn ngữ Anh",
  "Ngôn ngữ Bồ Đào Nha",
  "Ngôn ngữ Hàn Quốc",
  "Ngôn ngữ Italia",
  "Ngôn ngữ Nga",
  "Ngôn ngữ Nhật",
  "Ngôn ngữ Pháp",
  "Ngôn ngữ Trung Quốc",
  "Ngôn ngữ Tây Ban Nha",
  "Ngôn ngữ học",
  "Ngôn ngữ Đức",
  "Ngôn ngữ Ảrập",
  "Nha khoa (Răng - Hàm - Mặt)",
  "Nhân học",
  "Nhạc Jazz",
  "Nhật Bản học",
  "Nhật... (ngoại ngữ)",
  "Nuôi trồng thủy sản",
  "Nông học",
  "Năng lượng tái tạo",
  "Phát triển nông thôn",
  "Phòng cháy chữa cháy và cứu nạn cứu hộ",
  "Phục hồi chức năng",
  "Piano",
  "Quan hệ công chúng",
  "Quan hệ công chúng (PR)",
  "Quan hệ lao động",
  "Quan hệ quốc tế",
  "Quy hoạch",
  "Quy hoạch vùng và đô thị",
  "Quản lý",
  "Quản lý biển",
  "Quản lý bệnh viện",
  "Quản lý công",
  "Quản lý công nghiệp",
  "Quản lý dự án",
  "Quản lý giáo dục",
  "Quản lý hoạt động bay",
  "Quản lý kinh tế",
  "Quản lý nhà nước",
  "Quản lý năng lượng",
  "Quản lý phát triển đô thị và bất động sản",
  "Quản lý thông tin",
  "Quản lý thể dục thể thao",
  "Quản lý tài nguyên khoáng sản",
  "Quản lý tài nguyên nước",
  "Quản lý tài nguyên rừng",
  "Quản lý tài nguyên thiên nhiên",
  "Quản lý tài nguyên và môi trường",
  "Quản lý tài nguyên và môi trường biển đảo",
  "Quản lý và phát triển du lịch",
  "Quản lý và phân tích dữ liệu khoa học",
  "Quản lý và vận hành hạ tầng",
  "Quản lý văn hóa",
  "Quản lý xây dựng",
  "Quản lý đô thị và công trình",
  "Quản lý đất đai",
  "Quản trị - Luật",
  "Quản trị chất lượng giáo dục",
  "Quản trị công nghệ truyền thông",
  "Quản trị dịch vụ du lịch và lữ hành",
  "Quản trị khách sạn",
  "Quản trị kinh doanh",
  "Quản trị kinh doanh thực phẩm",
  "Quản trị nhà hàng và dịch vụ ăn uống",
  "Quản trị nhân lực",
  "Quản trị sự kiện",
  "Quản trị văn phòng",
  "Quốc tế học",
  "Robot và trí tuệ nhân tạo",
  "Răng - Hàm - Mặt",
  "Sinh dược học",
  "Sinh học",
  "Sinh học ứng dụng",
  "Sáng tác văn học",
  "Sáng tác âm nhạc",
  "Sư phạm (đào tạo giáo viên)",
  "Sư phạm Hóa học",
  "Sư phạm Khoa học tự nhiên",
  "Sư phạm Kỹ thuật nông nghiệp",
  "Sư phạm Lịch sử",
  "Sư phạm Lịch sử - Địa lý",
  "Sư phạm Mỹ thuật",
  "Sư phạm Ngữ văn",
  "Sư phạm Sinh học",
  "Sư phạm Tin học",
  "Sư phạm Tiếng Anh",
  "Sư phạm Tiếng Hàn Quốc",
  "Sư phạm Tiếng Nga",
  "Sư phạm Tiếng Nhật",
  "Sư phạm Tiếng Pháp",
  "Sư phạm Tiếng Trung Quốc",
  "Sư phạm Tiếng Đức",
  "Sư phạm Toán học",
  "Sư phạm Vật lý",
  "Sư phạm công nghệ",
  "Sư phạm Địa lý",
  "Thanh nhạc",
  "Thiên văn học",
  "Thiết kế Mỹ thuật số",
  "Thiết kế công nghiệp",
  "Thiết kế công nghiệp và Đồ họa",
  "Thiết kế nội thất",
  "Thiết kế thời trang",
  "Thiết kế vi mạch",
  "Thiết kế âm thanh - ánh sáng",
  "Thiết kế âm thanh, ánh sáng",
  "Thiết kế đồ họa",
  "Thông tin - Thư viện",
  "Thú y",
  "Thương mại điện tử",
  "Thống kê",
  "Thống kê kinh tế",
  "Thủy văn học",
  "Tiếng Việt và văn hóa Việt Nam",
  "Toán học",
  "Toán kinh tế",
  "Toán tin",
  "Toán ứng dụng",
  "Trinh sát kỹ thuật",
  "Triết học",
  "Trung",
  "Trung Quốc học",
  "Truyền thông doanh nghiệp",
  "Truyền thông quốc tế",
  "Truyền thông đa phương tiện",
  "Truyền thông đại chúng",
  "Trí tuệ nhân tạo",
  "Trí tuệ nhân tạo (AI)",
  "Trí tuệ nhân tạo và khoa học dữ liệu",
  "Tài chính - Ngân hàng",
  "Tài chính quốc tế",
  "Tài chính và Kế toán (BFA)",
  "Tài nguyên và Du lịch sinh thái",
  "Tâm lý học",
  "Tâm lý học giáo dục",
  "Tôn giáo học",
  "Tổ chức và Quản lý y tế",
  "Việt Nam học",
  "Văn hóa Việt Nam",
  "Văn hóa các dân tộc thiểu số Việt Nam",
  "Văn hóa học",
  "Văn học",
  "Vật lý học",
  "Vật lý kỹ thuật",
  "Vật lý y khoa",
  "Xuất bản",
  "Xây dựng công trình ngầm thành phố và Hệ thống tàu điện ngầm",
  "Xây dựng Đảng và chính quyền nhà nước",
  "Xã hội học",
  "Xét nghiệm",
  "Y học cổ truyền",
  "Y học dự phòng",
  "Y khoa",
  "Y khoa (bác sĩ)",
  "Y tế công cộng",
  "cơ điện tử",
  "dầu khí",
  "hóa học",
  "may",
  "môi trường",
  "nhà hàng",
  "sức khỏe và an toàn",
  "trắc địa - bản đồ",
  "tự động hóa",
  "vật liệu",
  "vệ sinh lao động",
  "Âm nhạc",
  "Âm nhạc học",
  "ô tô",
  "Điêu khắc",
  "Điều dưỡng",
  "Điện ảnh - truyền hình",
  "Đá quý Đá mỹ nghệ",
  "Đô thị học",
  "Đông Nam Á học",
  "Đông phương học",
  "Đạo diễn điện ảnh, truyền hình",
  "Đảm bảo chất lượng và an toàn thực phẩm",
  "Địa chất học",
  "Địa kỹ thuật xây dựng",
  "Địa lý học",
  "Địa lý tự nhiên",
  "Địa tin học",
  "Đồ họa",
  "điện tử",
  "điện tử - viễn thông",
  "Ẩm thực"
 ],
 "subjects": [
  "Toán",
  "Ngữ văn",
  "Tiếng Anh",
  "Vật lí",
  "Hóa học",
  "Sinh học",
  "Lịch sử",
  "Địa lí",
  "Giáo dục kinh tế và pháp luật",
  "Tin học",
  "Công nghệ công nghiệp",
  "Công nghệ nông nghiệp",
  "Khoa học tự nhiên",
  "Khoa học xã hội",
  "Tiếng Đức",
  "Tiếng Hàn",
  "Tiếng Nga",
  "Tiếng Nhật",
  "Tiếng Pháp",
  "Tiếng Trung"
 ]
}

TYPES = {
 "change_score": [
  "đổi điểm một hoặc vài môn thi của em",
  "subjects_scores"
 ],
 "change_priority": [
  "đổi khu vực ưu tiên (KV1, KV2-NT, KV2, KV3) hoặc đối tượng ưu tiên (ưu tiên 1 / nhóm đối tượng 01-03, ưu tiên 2 / nhóm 04-06, hoặc không ưu tiên)",
  "priority"
 ],
 "change_budget": [
  "đổi ngân sách học phí mỗi năm (triệu đồng), hoặc bỏ giới hạn ngân sách",
  "budget"
 ],
 "change_near_home": [
  "bật hoặc tắt mong muốn học gần nhà",
  "near_home"
 ],
 "extra_points": [
  "nhập điểm cộng mà một trường cụ thể cho em (em đã tự tính theo quy định của trường)",
  "school_points"
 ],
 "favorite_school": [
  "chọn một trường là trường em thích nhất / muốn vào nhất (để trường đó được ưu tiên trong danh sách), hoặc bỏ trường yêu thích đã chọn",
  "school_or_none"
 ],
 "own_score": [
  "nhập điểm xét tuyển của em ở một trường, em đã tự tính theo cách tính riêng của trường đó (ví dụ trường xét kết hợp điểm thi với học bạ hoặc điểm đánh giá năng lực, thang 30 hoặc thang 100)",
  "school_own_score"
 ],
 "cutoff_shift": [
  "nói rằng điểm chuẩn của một trường, với em, thực ra cao hơn hoặc thấp hơn con số công bố (ví dụ trường đó có nhiều điểm cộng nên điểm chuẩn bị đẩy lên), để ứng dụng điều chỉnh cho riêng em",
  "school_shift"
 ],
 "remove": [
  "bỏ một trường (mọi chương trình của trường) hoặc một nguyện vọng theo số thứ tự NV khỏi danh sách",
  "school_or_nv"
 ],
 "restore": [
  "đưa lại vào danh sách một trường hoặc nguyện vọng em đã bỏ trước đó",
  "school_or_removed"
 ],
 "rerank": [
  "đổi thứ tự hai lựa chọn ngành của em (lựa chọn 2 lên trước lựa chọn 1)",
  "none"
 ],
 "replace_nganh": [
  "thay một lựa chọn ngành bằng ngành khác, hoặc thêm / bỏ / đổi một ngành gần của lựa chọn",
  "two_nganh"
 ],
 "undo": [
  "hoàn tác thay đổi vừa làm",
  "none"
 ],
 "ask_why_nv": [
  "hỏi vì sao một chương trình lại nằm ở nguyện vọng số N trong danh sách",
  "nv"
 ],
 "ask_why_not": [
  "hỏi vì sao một trường không có trong danh sách nguyện vọng",
  "school"
 ],
 "ask_compare": [
  "hỏi hai ngành khác nhau thế nào / nên chọn ngành nào",
  "two_nganh"
 ],
 "ask_what_is": [
  "hỏi về một ngành: học gì, có khó không, cần giỏi môn nào, ra trường làm gì, thu nhập",
  "nganh"
 ],
 "ask_level": [
  "hỏi một mức trong danh sách (an toàn, vừa sức, thử sức, mơ ước) nghĩa là gì",
  "level"
 ],
 "other": [
  "nói chuyện ngoài lề hoặc nhờ việc ứng dụng không làm (chào hỏi, cảm ơn, hỏi dự đoán điểm chuẩn năm nay, xin số điện thoại, nhờ làm bài tập, tâm sự)",
  "none"
 ]
}

SEEDS = {
 "change_score": [
  "toán em 8,25 nha",
  "sửa điểm hóa thành 7",
  "anh van e dc 9.5"
 ],
 "change_priority": [
  "em ở KV2-NT",
  "nhà em thuộc khu vực 1",
  "em không có đối tượng ưu tiên"
 ],
 "change_budget": [
  "học phí tối đa 35 triệu/năm",
  "nhà em lo được tầm 20tr thôi",
  "không giới hạn học phí"
 ],
 "change_near_home": [
  "em muốn học gần nhà",
  "đi xa cũng được ạ",
  "ko cần gần nhà"
 ],
 "extra_points": [
  "trường Thương mại cộng cho em 1 điểm",
  "UEH cộng 0.75",
  "em được Bách khoa cộng 2 điểm IELTS"
 ],
 "favorite_school": [
  "trường em thích nhất là Ngoại thương",
  "em muốn vào Bách khoa nhất",
  "bỏ trường yêu thích đi"
 ],
 "own_score": [
  "điểm xét tuyển của em ở Bách khoa TP.HCM là 82",
  "em tự tính điểm ở UIT được 27,5",
  "theo cách tính của trường Kinh tế Quốc dân thì e được 28"
 ],
 "cutoff_shift": [
  "điểm chuẩn UIT nhiều điểm cộng lắm, thực ra thấp hơn 1 điểm",
  "điểm chuẩn NEU cao hơn thực tế 0,5",
  "30 điểm của trường Ngoại thương không cao bằng 29 của Bách khoa đâu"
 ],
 "remove": [
  "bỏ trường Văn Lang",
  "xoá NV4 giúp em",
  "em ko muốn học Ngoại thương"
 ],
 "restore": [
  "cho Văn Lang vào lại",
  "lấy lại trường em vừa bỏ",
  "đưa NEU trở lại"
 ],
 "rerank": [
  "đổi thứ tự 2 ngành",
  "cho lựa chọn 2 lên đầu",
  "hoán đổi hai lựa chọn"
 ],
 "replace_nganh": [
  "thay Kế toán bằng Kiểm toán",
  "thêm ngành gần Marketing",
  "đổi lựa chọn 1 sang Luật"
 ],
 "undo": [
  "hoàn tác",
  "quay lại như trước",
  "huỷ cái vừa đổi"
 ],
 "ask_why_nv": [
  "sao NV2 lại là trường này",
  "vì sao nguyện vọng 9 ở đó",
  "giải thích NV1"
 ],
 "ask_why_not": [
  "sao không có Bách khoa",
  "NEU đâu rồi",
  "vì sao thiếu trường Ngoại thương"
 ],
 "ask_compare": [
  "Kế toán với Kiểm toán khác gì",
  "nên học Luật hay Luật kinh tế",
  "so sánh CNTT và KHMT"
 ],
 "ask_what_is": [
  "ngành Logistics học gì",
  "Marketing ra làm gì",
  "Dược học có khó không"
 ],
 "ask_level": [
  "thử sức là gì",
  "mức an toàn nghĩa là chắc đỗ à",
  "vừa sức là sao"
 ],
 "other": [
  "chào bạn",
  "điểm chuẩn năm nay bao nhiêu",
  "giúp em làm bài toán này"
 ]
}

ATTRIBUTES = {
 "writer": [
  "học sinh lớp 12",
  "học sinh lớp 12",
  "phụ huynh"
 ],
 "register": [
  "thân mật, nói như nhắn tin với bạn",
  "lịch sự, có 'ạ', 'dạ'",
  "ngắn gọn, ra lệnh"
 ],
 "accents": [
  "có dấu đầy đủ",
  "có dấu đầy đủ",
  "không dấu (telex chưa bật)",
  "lẫn lộn chỗ có dấu chỗ không"
 ],
 "noise": [
  "không lỗi",
  "vài lỗi gõ phím",
  "dùng từ viết tắt kiểu teencode (k, ko, dc, e, j, ntn)"
 ],
 "length": [
  "rất ngắn (dưới 6 từ)",
  "một câu ngắn",
  "một câu dài, có kể thêm hoàn cảnh"
 ],
 "form": [
  "câu khẳng định hoặc mệnh lệnh",
  "câu hỏi",
  "kể / tường thuật"
 ]
}

## Generation

In [ ]:
random.seed(SEED)
LEVELS = ["an toàn", "vừa sức", "thử sức", "mơ ước"]

def score_text(rng):
    v = rng.choice([x / 4 for x in range(8, 41)])
    words = {5: "năm", 6: "sáu", 7: "bảy", 8: "tám", 9: "chín", 10: "mười"}
    if rng.random() < 0.15 and v == int(v) and int(v) in words:
        return words[int(v)]
    s = f"{v:g}"
    return s.replace(".", ",") if rng.random() < 0.5 else s

def slot_hint(kind, rng):
    # Values the messages should use, from UniAdvisor's real data (names) or random (numbers).
    if kind == "subjects_scores":
        subs = rng.sample(SLOTS["subjects"], rng.choice([1, 1, 2, 3]))
        return "môn và điểm: " + "; ".join(f"{s} {score_text(rng)}" for s in subs)
    if kind == "priority":
        return "giá trị: " + rng.choice(["KV1", "KV2-NT", "KV2", "KV3", "ưu tiên 1", "ưu tiên 2", "nhóm đối tượng 03",
                                         "nhóm đối tượng 06", "không thuộc đối tượng ưu tiên"])
    if kind == "budget":
        return "ngân sách: " + rng.choice([f"{rng.randrange(10, 101, 5)} triệu/năm", "không giới hạn",
                                           f"{rng.randrange(10, 101, 5)}tr"])
    if kind == "near_home":
        return "ý muốn: " + rng.choice(["muốn học gần nhà", "không cần học gần nhà"])
    if kind == "school_points":
        return f"trường: {rng.choice(SLOTS['schools'])}; điểm cộng: {rng.choice(['0,25', '0,5', '0.75', '1', '1,5', '2', '3'])}"
    if kind == "school_or_none":
        return rng.choice([f"trường: {rng.choice(SLOTS['schools'])}"] * 4 + ["bỏ trường yêu thích (không nói tên)"])
    if kind == "school_own_score":
        on_30 = f"{rng.randrange(20, 30)},{rng.choice(['0', '25', '5', '75'])}"
        score = rng.choice([f"{rng.randrange(55, 98)}/100", f"{rng.randrange(55, 98)} (thang 100)",
                            f"{on_30} (thang 30)", on_30])
        return f"trường: {rng.choice(SLOTS['schools'])}; điểm xét tuyển em tự tính: {score}"
    if kind == "school_shift":
        how = rng.choice(["thấp hơn", "cao hơn", "thấp hơn", "không nói số điểm, chỉ nói cao hơn / thấp hơn"])
        return (f"trường: {rng.choice(SLOTS['schools'])}; điểm chuẩn với em {how}"
                + ("" if how.startswith("không") else f" {rng.choice(['0,5', '1', '1,5', '2', '0.25', '3'])} điểm"))
    if kind == "school_or_nv":
        return rng.choice([f"trường: {rng.choice(SLOTS['schools'])}", f"nguyện vọng số {rng.randint(1, 15)}"])
    if kind == "school_or_removed":
        return rng.choice([f"trường: {rng.choice(SLOTS['schools'])}", "trường / nguyện vọng em vừa bỏ (không nói tên)"])
    if kind == "two_nganh":
        a, b = rng.sample(SLOTS["nganh"], 2)
        return f"hai ngành: {a}; {b}"
    if kind == "nganh":
        return f"ngành: {rng.choice(SLOTS['nganh'])}"
    if kind == "nv":
        return f"nguyện vọng số {rng.randint(1, 15)}"
    if kind == "school":
        return f"trường: {rng.choice(SLOTS['schools'])}"
    if kind == "level":
        return f"mức: {rng.choice(LEVELS)}"
    return ""

def build_prompt(rtype, rng):
    desc, kind = TYPES[rtype]
    attrs = {k: rng.choice(v) for k, v in ATTRIBUTES.items()}
    hint = slot_hint(kind, rng)
    seeds = "\n".join(f"- {s}" for s in SEEDS[rtype])
    prompt = (
        "Bạn giúp tạo dữ liệu huấn luyện cho chatbot của một ứng dụng tư vấn đăng ký nguyện vọng đại học ở Việt Nam. "
        f"Hãy viết đúng {MESSAGES_PER_PROMPT} tin nhắn khác nhau mà {attrs['writer']} có thể gõ vào khung chat, "
        f"mỗi tin nhắn chỉ có MỘT yêu cầu thuộc loại: {desc}.\n"
        f"Ví dụ về loại này:\n{seeds}\n"
        + (f"Dùng các giá trị sau (có thể viết tắt hoặc gọi theo cách thường nói): {hint}\n" if hint else "")
        + f"Phong cách: {attrs['register']}; {attrs['accents']}; {attrs['noise']}; độ dài: {attrs['length']}; "
        f"dạng: {attrs['form']}.\n"
        "Không lặp lại ví dụ. Không thêm yêu cầu thuộc loại khác. "
        "Chỉ trả về một mảng JSON gồm các chuỗi, không giải thích."
    )
    return prompt, attrs, hint

def parse(text):
    m = re.search(r"\[.*\]", text, re.S)
    if not m:
        return []
    try:
        items = json.loads(m.group(0))
    except json.JSONDecodeError:
        return []
    return [s.strip() for s in items if isinstance(s, str) and 2 <= len(s.strip()) <= 300]

def load(model_id, how):
    tok = AutoTokenizer.from_pretrained(model_id, token=os.environ.get("HF_TOKEN"))
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kwargs = dict(device_map="auto", token=os.environ.get("HF_TOKEN"))
    if how == "fp16":
        kwargs["torch_dtype"] = torch.float16
    elif how == "4bit_fp32":
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float32)
        kwargs["attn_implementation"] = "eager"
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    return tok, model

def chat_text(tok, prompt):
    messages = [{"role": "user", "content": prompt}]   # no system role: Gemma has none
    if tok.chat_template:
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Fallback when the tokenizer ships no template: SeaLLM-7B-v2's documented format (one BOS, no newline after </s>).
    return f"<s><|im_start|>user\n{prompt}</s><|im_start|>assistant\n"

def run_family(family):
    model_id, how = MODELS[family]
    rng = random.Random(f"{SEED}-{family}")
    jobs = []
    for rtype in (RUN_TYPES or TYPES):
        for k in range(PROMPTS_PER_TYPE):
            prompt, attrs, hint = build_prompt(rtype, rng)
            jobs.append((f"{family}-{rtype}-{k}", rtype, prompt, attrs, hint))
    tok, model = load(model_id, how)
    rows, start = [], time.time()
    for b in range(0, len(jobs), BATCH_SIZE):
        batch = jobs[b:b + BATCH_SIZE]
        enc = tok([chat_text(tok, j[2]) for j in batch], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE,
                                 top_p=TOP_P, pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for (pid, rtype, prompt, attrs, hint), text in zip(batch, texts):
            for msg in parse(text):
                rows.append({"text": msg, "type": rtype, "family": family, "model": model_id,
                             "attributes": json.dumps({**attrs, "values": hint}, ensure_ascii=False),
                             "prompt_id": pid})
        if b % (BATCH_SIZE * 10) == 0:
            print(f"{family}: {b + len(batch)}/{len(jobs)} prompts, {len(rows)} messages, {time.time() - start:.0f}s")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    tag = family if ROUND == 1 else f"{family}_round{ROUND}"
    df.to_csv(f"/kaggle/working/chat_generated_{tag}.csv", index=False)
    (df.groupby("type", group_keys=False).apply(lambda g: g.sample(min(20, len(g)), random_state=SEED))
       .to_csv(f"/kaggle/working/chat_review_sample_{tag}.csv", index=False))
    print(family, "done:", len(df), "messages")
    print(df["type"].value_counts())
    return df

results = {f: run_family(f) for f in RUN_FAMILIES}

## After the run

1. Download `chat_generated_<family>.csv` into `pipelines/data/chat/generated/` and `chat_review_sample_<family>.csv` into
   `pipelines/data/chat/review/` of the repository.
2. People read the review samples and mark wrong or unnatural messages.
3. `python -m pipelines.chat_model.train` trains on the training families, tests on the held-out family and on the people-
   judged check set, and writes `models/message_classifier/report.md`. The app switches from rules to the model only
   when the model reaches the targets in `config/chat.yaml` and beats the rules.